# PolicyDiff: English–Hindi fine-tuning
Fine-tune a pretrained encoder–decoder on a small IIT Bombay sample, track validation loss, and compare both models on one fixed test set.

Select **Runtime → Change runtime type → T4 GPU** before running. Defaults: 3,000 training pairs, 300 validation pairs, 500 test pairs, two epochs. Free GPU availability varies. No results have been pre-filled.

Use validation loss to select the checkpoint. Do not tune settings based on test BLEU. Training and inference remain in this Colab runtime; no weights are published automatically.

In [ ]:
from pathlib import Path
import subprocess
import sys
repo = Path('/content/PolicyDiff')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AmitTiparadi/PolicyDiff.git', str(repo)], check=True)
%cd /content/PolicyDiff
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-training.txt'], check=True)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## Configure the experiment
Choose these settings before evaluating the test set. Use `EPOCHS = 1`, `2`, or `3`. The run refuses to overwrite an existing output directory. If you need a new run, change `RUN_NAME`. For a GPU memory error, try `BATCH_SIZE = 2` and `ACCUMULATION_STEPS = 8` in a fresh run.

The sample excludes exact normalized English or Hindi overlap between splits. Long examples are excluded rather than silently truncated. The official dataset revision and exact selected rows are recorded. Original pretraining overlap is unknown.

In [ ]:
RUN_NAME = 'iitb_3000_seed42'
TRAIN_SIZE = 3000
VALIDATION_SIZE = 300
TEST_SIZE = 500
EPOCHS = 2
BATCH_SIZE = 4
ACCUMULATION_STEPS = 4
output = repo / 'outputs' / RUN_NAME
command = [sys.executable, '-m', 'scripts.finetune_translation',
           '--output', str(output), '--train-size', str(TRAIN_SIZE),
           '--validation-size', str(VALIDATION_SIZE), '--test-size', str(TEST_SIZE),
           '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
           '--accumulation-steps', str(ACCUMULATION_STEPS)]
subprocess.run(command, cwd=repo, check=True)


## Inspect measured results
BLEU and chrF measure reference overlap, not legal adequacy. A positive, zero, or negative BLEU difference is a valid experimental result. Investigate overfitting, sample size, and domain mismatch using the validation curve and individual translations; do not assert a cause from BLEU alone. These general-domain results do not replace privacy-policy evaluation.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image, Markdown
results = json.loads((output / 'comparison.json').read_text())
assert results['status'] == 'completed'
display(pd.read_csv(output / 'comparison.csv'))
print('BLEU difference:', results['bleu_delta'])
display(Image(filename=str(output / 'loss_curve.png')))
display(Image(filename=str(output / 'bleu_comparison.png')))
display(pd.read_csv(output / 'test_predictions.csv').head(10))
display(Markdown((output / 'RESULTS.md').read_text()))


## Download experiment evidence
Download before disconnecting: `/content` is temporary. This ZIP contains actual metrics, plots, configuration, provenance, and sampled data. Large model/checkpoint directories are excluded. The app still uses its pretrained model until fine-tuned weights are reviewed and integrated.

In [ ]:
import zipfile
from google.colab import files
archive = Path('/content') / f'{RUN_NAME}_evidence.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(output.iterdir()):
        if path.is_file():
            z.write(path, path.name)
files.download(str(archive))


## Optional: retain the selected model
Run this cell if you want the fine-tuned weights. The download may be several hundred MB. Intermediate training checkpoints are not included. Save it before the runtime expires.

In [ ]:
# Uncomment to download the best model and tokenizer.
# import shutil
# model_zip = shutil.make_archive(f'/content/{RUN_NAME}_best_model', 'zip', output / 'best_model')
# files.download(model_zip)
